# 1-D spherical TI, frequency domain — DSM-style SH (toroidal) with flexOPT

Goal: reproduce, inside flexOPT, the spherically symmetric DSM computations of

* Cummins, Geller, Hatori & Takeuchi (1994a), GRL — `94GL00013` — SH case
* Cummins, Geller & Takeuchi (1994b), GRL — `94GL01281` — P-SV case
* Takeuchi, Geller & Cummins (1996), GRL — `96GL00973` — modified (optimally accurate) P-SV operators

for **TI media** (Love's A, C, F, L, N, radial symmetry axis), then go beyond them with
the flexOPT B-spline machinery (FD, OPT3, OPT4, OPT5), free surface and overlapping
operators at each discontinuity. SH first; P-SV later (fluid outer core, and then
atmosphere + ionosphere).

### Roadmap (each step = stop-and-review point)

| step | content | where |
|---|---|---|
| **1** | symbolic: Cartesian TI equation → normalised toroidal harmonics → project with $C^*_{lm}$ → radial strong form with $\Lambda=l(l+1)$ → `famousEquation` entry | **this notebook (now)** |
| 2 | semi-symbolic recipes (`makeOPTsemiSymbolic`) for FD3/OPT3/OPT4/OPT5 on the new entry; operator-level comparison with the DSM $T$, $H$ matrices for a homogeneous layer (as `scripts/compareTakeuchiGellerRecette.jl` does for the Cartesian case) | here |
| 3 | homogeneous solid sphere, free surface: toroidal eigenfrequencies $(l-1)j_l(kR)=kR\,j_{l+1}(kR)$ and forced response vs $j_l(kr)$; convergence FD/OPT3/4/5; centre treatment | here |
| 4 | discontinuities: two-layer sphere, overlapping operators at the interface (`overlapBoundaryLinearSystem`), convergence | here |
| 5 | PREM TI ($L=\rho V_{sv}^2$, $N=\rho V_{sh}^2$) from `planet1D`, sum over $l$, point source, seismograms; comparison with DSM/MINEOS | → `scripts/` |
| 6 | P-SV (spheroidal), fluid outer core, atmosphere/ionosphere | later |

### Conventions used here
* $\rho\,\omega^2 u_i + \partial_j\!\left(C_{ijkl}\,\partial_l u_k\right) = -f_i$ (the sign of the source side is checked in Step 2, as in `seismo1Dbenchmark.ipynb`).
* $Y_{lm}$ fully normalised, $\int_{S^2} Y_{lm}Y^*_{l'm'}\,d\Omega=\delta_{ll'}\delta_{mm'}$.
* toroidal vector harmonic $\mathbf C_{lm} = -k^{-1}\,\hat{\mathbf r}\times\nabla_1 Y_{lm}$, $k=\sqrt{l(l+1)}$, so $\int \mathbf C_{lm}\cdot\mathbf C^*_{l'm'}\,d\Omega=\delta\delta$.
* $\mathbf u = W(r)\,\mathbf C_{lm}$; projection = multiply by $\mathbf C^*_{lm}$ from the left and integrate over $d\Omega$.
* the radial coordinate is flexOPT's `x` (exported by `commonBatchs`): **x ≡ r**.

Code: `helpers/sphericalTI_SH.jl` (Claude-authored; `src/` is untouched — the new
`famousEquation` methods are added from outside).

In [ ]:
# Same bootstrap as seismo1Dbenchmark.ipynb
import Pkg

function find_flexopt_root(start_dir=pwd())
    candidates = String[]
    if haskey(ENV, "FLEXOPT_ROOT")
        push!(candidates, abspath(expanduser(ENV["FLEXOPT_ROOT"])))
    end
    directory = abspath(start_dir)
    while true
        push!(candidates, directory)
        parent = dirname(directory)
        parent == directory && break
        directory = parent
    end
    for candidate in unique(candidates)
        project_file = joinpath(candidate, "Project.toml")
        source_dir = joinpath(candidate, "src")
        if isfile(project_file) && isfile(joinpath(source_dir, "commonBatchs.jl"))
            return candidate
        end
    end
    error("Cannot locate flexOPT. Start Jupyter inside the repository or set ENV[\"FLEXOPT_ROOT\"] to its absolute path.")
end

flexopt_root = find_flexopt_root()
Pkg.activate(flexopt_root)
@show VERSION Threads.nthreads() Base.active_project()

# Metal must be loaded before batchGPU.jl selects the backend.
using Metal
Metal.functional() || error("Metal.jl is loaded, but cannot access the Apple GPU")

include(joinpath(flexopt_root, "src", "batchFiles", "batchGPU.jl"))
include(joinpath(flexopt_root, "src", "commonBatchs.jl"))
include(joinpath(flexopt_root, "src", "planet1D.jl"))
planet1D.configure_input!()
include(joinpath(flexopt_root, "src", "flexOPT.jl"))
using .commonBatchs, .planet1D, .flexOPT

using Symbolics, LinearAlgebra, Random
include(joinpath(flexopt_root, "helpers", "sphericalTI_SH.jl"))

## Step 1a — the Cartesian TI equation, symbolically

$$\rho\,\omega^2 u_i + \partial_j\!\left(C_{ijkl}\,\partial_l u_k\right) = -f_i,\qquad
C_{ijkl}=(A-2N)\delta_{ij}\delta_{kl}+N(\delta_{ik}\delta_{jl}+\delta_{il}\delta_{jk})
+(F-A+2N)(\delta_{ij}n_kn_l+n_in_j\delta_{kl})$$
$$\qquad+(L-N)(\delta_{ik}n_jn_l+\delta_{il}n_jn_k+\delta_{jk}n_in_l+\delta_{jl}n_in_k)+(A+C-2F-4L)\,n_in_jn_kn_l,\quad n=\mathbf x/|\mathbf x|.$$

First two sanity checks on the tensor (isotropic limit, Love notation for $n=\hat z$), then the
full equation as a `famousEquation` entry `"3DsismoFreqTIradial"` (kept for display/reference).

In [ ]:
@show checkTIisotropicLimitAndLoveNotation()

eq3D, force3D = famousEquations("3DsismoFreqTIradial")
@show eq3D.fields eq3D.vars eq3D.coordinates
# the expressions are big (n = x/|x| inside C_ijkl): show the size and the beginning
for i in 1:3
    s = string(eq3D.exprs[i])
    println("component $i : ", length(s), " characters\n   ", first(s, 400), " …\n")
end

## Step 1b–1c — spherical components, normalised toroidal harmonic, projection

Insert $\mathbf u = W(r)\mathbf C_{lm}$ in $\rho\omega^2\mathbf u+\nabla\cdot\boldsymbol\sigma$ written in spherical physical components
(TI stress with axis $\hat{\mathbf r}$), then multiply by $\mathbf C^*_{lm}$ and integrate over $d\Omega$.

Because the operator is rotationally invariant, the result cannot depend on $m$, so the
symbolic derivation is done with $m=0$ ($\mathbf u=(0,0,-W P')$ up to the constant $k^{-1}N_{l0}$, which cancels).
The angular dependence is carried by a *jet*: $q=\cot\theta$ with $q'=-(1+q^2)$, $p_0=P$, $p_1=P'$ with
the Legendre ODE $P''=-qP'-\Lambda P$. Everything then cancels by polynomial `expand` alone.
The result is $\rho\omega^2\mathbf u+\nabla\cdot\boldsymbol\sigma = \mathcal R(r)\,\mathbf C_{lm}$, i.e.
$\int\mathbf C^*_{lm}\cdot(\dots)\,d\Omega=\mathcal R(r)$, and we expect (Takeuchi & Saito toroidal system, TI)

$$\mathcal R = \omega^2\rho W + \frac{1}{r^3}\frac{d}{dr}\!\Big[r^3 L\Big(W'-\frac{W}{r}\Big)\Big]-(\Lambda-2)\frac{N\,W}{r^2} = -f_W(r),\qquad \Lambda=l(l+1).$$

Only $L$ and $N$ survive (A, C, F drop out for SH, as they should).

In [ ]:
sym = deriveToroidalSymbolic()
println("E_r  = ", sym.E[1])
println("E_θ  = ", sym.E[2])
println("\nR(r) = ∫ C*·(ρω²u + ∇·σ) dΩ =\n", sym.R)
println("\nexpected =\n", sym.expected)
println("\nsymbolic R - expected (expanded, ×r³) = ",
        Symbolics.expand(sym.R * x^3 - sym.expected * x^3))

In [ ]:
# Numerical verification with concrete radial profiles at random (r, cotθ, P, P'),
# several l: E_r = E_θ = 0, no P term, no θ dependence of R, R = expected,
# and the self-adjoint (flexOPT) form = r² R.
for l in (1, 2, 5, 20)
    @show l verifyToroidalProjection(; l=l)
end

### Independent check with $m\neq0$ — brute force in Cartesian coordinates

Full Cartesian TI tensor with axis $\mathbf x/|\mathbf x|$, real toroidal fields built from
$Y\propto(x^2-y^2)z/r^3$ ($l=3,m=2$) and $Y\propto xz/r^2$ ($l=2,m=1$), operator applied component by
component and compared with $\mathcal R(r)\,\mathbf C(\theta,\phi)$ at random points.
This closes the loop Step 1a → Step 1c without using the spherical formulas at all.
(It can take a minute or two — the Cartesian expression is large.)

In [ ]:
@time @show cartesianToroidalCheck(; harmonic=:Y21)
@time @show cartesianToroidalCheck(; harmonic=:Y32)

## Step 1d — the radial equation as a `famousEquation` entry

Multiplying by $r^2$ and writing it in self-adjoint form:

$$\partial_r(r^2L\,\partial_rW)-\partial_r(rL\,W)+rL\,\partial_rW+\big(\omega^2\rho r^2-[L+(\Lambda-2)N]\big)W=-r^2f_W .$$

Entry `"1DsphericalSH_TI_Freq"` uses **geometric material fields**
$\rho r^2,\ Lr^2,\ Lr,\ N_l=L+(\Lambda-2)N$ and the source $f r^2$, because

* `PDECoefFinder` assumes every coefficient is (material field) × (field derivative), with no explicit coordinate;
* `naturalWeakForm` integrates by parts only `const × ∂x(flux)`;
* folding $\Lambda$ into $N_l$ keeps the semi-symbolic recipe **independent of $l$** (one recipe for all $l$; only the material arrays change).

Its natural weak form should be exactly the DSM SH operator $\omega^2T-H$:
$T_{ij}=\int\rho r^2X_iX_j\,dr$, $H_{ij}=\int L(rX_i'-X_i)(rX_j'-X_j)\,dr+(\Lambda-2)\int N X_iX_j\,dr$,
with boundary flux $-r^2L(W'-W/r)$, which is zero at a free surface (natural condition).

In [ ]:
eqSH, forceSH = famousEquations("1DsphericalSH_TI_Freq")
@show eqSH.exprs eqSH.fields eqSH.vars eqSH.coordinates forceSH.exprs

wf = naturalWeakForm(eqSH)
println("\nvolume terms (expression, derivative order on the test function):")
for t in wf.volume_terms[1]
    println("   ", t.test_derivative_orders, "  =>  ", t.expression)
end
println("\nboundary fluxes (normal coordinate, expression):")
for b in wf.boundary_fluxes[1]
    println("   ", b.normal_coordinate, "  =>  ", b.expression)
end
println("\nexpected: (1,) => -Lr²∂W + Lr W ;  (0,) => Lr ∂W, ω²ρr² W, -Nₗ W")

In [ ]:
# Check that the geometric fields reproduce the radial operator:
# substitute ρr²→ρr², Lr²→Lr², Lr→Lr, Nₗ→L+(Λ-2)N in the entry and compare with r²·expected.
let s = sym.symbols
    @variables ρr²(x) Lr²(x) Lr(x) Nₗ(x) ω
    e = substitute(eqSH.exprs, Dict(ρr² => s.ρ * x^2, Lr² => s.L * x^2, Lr => s.L * x,
                                    Nₗ => s.L + (s.Λ - 2) * s.N))
    e = substitute(e, Dict(ω => s.ω))
    diff = Symbolics.expand(expand_derivatives(e) - x^2 * sym.expected)
    # Symbolics does not distribute the 1/x of the expected form: clear it before expanding
    println("x·(entry(geometric fields) - r²·expected) = ", Symbolics.expand(x * diff))
end

## ⏸ Stop and review (end of Step 1)

Please check:
1. `checkTIisotropicLimitAndLoveNotation()` → both `true`;
2. `verifyToroidalProjection` discrepancies ~1e-13 or smaller for all $l$;
3. `cartesianToroidalCheck` relative error ~1e-13 or smaller for both harmonics;
4. the `naturalWeakForm` terms match the DSM $T$/$H$ structure.

Points for you to decide before Step 2:
* the scalar `ω` is kept as a `var`, as in `eq_1DsismoFreqHetero`; I still need to check how the numerical
  operator builder expects a scalar (constant material array or separate). Alternatively ω²ρr² could be one material field;
* which source representation you want for Step 3 (body-force $f_W$ vs. the DSM point-source jump in traction at $r_s$);
* centre: start the grid at $r_{\min}>0$ with $W=0$ (DSM cut-off below the turning point) or go to $r=0$ with $W\sim r^l$.

---
# Step 2 — flexOPT operators (FD3, OPT3, OPT4, OPT5) vs conventional DSM, on planet1D models

Reference = **conventional DSM SH**: linear splines, element-by-element Gauss integration (exact for planet1D's cubics),
in $\xi=r/R$ with $\Omega=\omega R$:
$T_{ij}=\int\rho\xi^2X_iX_j\,d\xi$, $H_{ij}=\int L(\xi X_i'-X_i)(\xi X_j'-X_j)\,d\xi+(\Lambda-2)\int N X_iX_j\,d\xi$, $K=\Omega^2T-H$.
The **modified (optimally accurate) spherical operators** of Cummins et al. (1994a) will be added as a second reference once their equations are in hand.

Design choices:
* **models from `src/planet1D`**: in-memory DSM-SH models (`layeredSHModel`, same constructor as the model files) and PREM (`getSet1Dmodel!`). $L=\rho V_{sv}^2$, $N=\rho V_{sh}^2$ from the zone polynomials. SH sees only the outermost continuous solid region.
* **grids**: `mode=:uniform` (default) — one uniform $h$ **per zone**, $h_{zone}=\text{width}/\lceil\text{width}/h\rceil$, so $h$ changes at every discontinuity; `mode=:dsm` — `planet1D.VerticalGridStructure` (GT95 eq. 6.2, `re`, $\omega_{max}$). Interface radii are duplicated (top of the lower zone, bottom of the upper zone) for Step 4.
* **scaled local coordinate** per zone, $s=(\xi-\xi_{bot})/h$: one recipe at $\Delta=1$ for all zones and all $h$; the powers of $h$ go into the material fields: $(\rho r^2, Lr^2, Lr, N_l, \omega)\to(h^2\rho\xi^2,\ \xi^2L,\ h\xi L,\ h^2N_l,\ \Omega)$. Expected: $A\approx h\,K$ (printed as `s/h`).
* **ω** is a scalar model (`prepareNumericalOperatorGeometry` treats a var without x-dependence as a constant).

Measures (interior rows only, away from the segment ends):
* `s/h` and $\|A-sK\|/\|sK\|$ — how close each flexOPT operator is to conventional DSM;
* `res` $=\max|A\,W_{exact}|/\max|A_{\Omega^2}W_{exact}|$ (Geller–Takeuchi criterion), $W_{exact}=j_l(\Omega\xi/\beta)$ for a homogeneous isotropic shell. Conventional DSM and FD3 are $O(h^2)$; OPT3 is also $O(h^2)$ but with a much smaller error constant, so the relevant number is the **ratio to convFD3 at the same h** (printed after the convergence runs).

In [ ]:
include(joinpath(flexopt_root, "helpers", "sphericalTI_SH_step2.jl"))
using CairoMakie
CairoMakie.activate!()

### 2a — configurations and recipes
Same definitions as `seismo1Dbenchmark.ipynb` (FD3 uses the strong form automatically, the OPTs the natural weak form).
Recipes are built once at $\Delta=1$ and cached (`data/semiSymbolic`); the first run takes a while.

In [ ]:
shConfigs = NamedTuple[]
push!(shConfigs, (name="convFD3", orderBtime=-1, orderBspace=-1, pointsInSpace=3, pointsInTime=3, supplementaryOrder=0,
    fieldItpl=(ptsSpace=1, ptsTime=1, offsetSpace=1.0, offsetTime=1, YorderBspace=-1, YorderBtime=-1),
    materItpl=(ptsSpace=1, ptsTime=1, offsetSpace=1.0, offsetTime=1, YorderBspace=-1, YorderBtime=-1)))
push!(shConfigs, (name="OPT3", orderBtime=1, orderBspace=1, pointsInSpace=3, pointsInTime=3, supplementaryOrder=2,
    fieldItpl=(ptsSpace=1, ptsTime=1, offsetSpace=1.0, offsetTime=1, YorderBspace=-1, YorderBtime=-1),
    materItpl=(ptsSpace=1, ptsTime=1, offsetSpace=1.0, offsetTime=1, YorderBspace=-1, YorderBtime=-1)))
push!(shConfigs, (name="OPT4", orderBtime=1, orderBspace=1, pointsInSpace=4, pointsInTime=1, supplementaryOrder=2,
    fieldItpl=(ptsSpace=1, ptsTime=1, offsetSpace=1.5, offsetTime=1, YorderBspace=-1, YorderBtime=1),
    materItpl=(ptsSpace=1, ptsTime=1, offsetSpace=1.5, offsetTime=1, YorderBspace=-1, YorderBtime=1)))
push!(shConfigs, (name="OPT5", orderBtime=1, orderBspace=1, pointsInSpace=5, pointsInTime=1, supplementaryOrder=2,
    fieldItpl=(ptsSpace=1, ptsTime=1, offsetSpace=2.0, offsetTime=1, YorderBspace=-1, YorderBtime=1),
    materItpl=(ptsSpace=1, ptsTime=1, offsetSpace=2.0, offsetTime=1, YorderBspace=-1, YorderBtime=1)))

shRecipes = Dict{String,Any}()
for c in shConfigs
    @time shRecipes[c.name] = shRecipe(c)
    r = shRecipes[c.name]["recette"]
    println(c.name, ": form=", r.form, "  varM rows=", size(r.lhs.varM, 1),
            "  stencil nodes=", length(r.nodes[1]), "  Ajiννᶜ size=", size(r.lhs.Ajiννᶜ))
end

### 2b-i — a single grid first (homogeneous isotropic shell, $r_{cmb}\le r\le R$)
Sanity check before any convergence study: matrix sizes, `s/h` (should be ≈ ±1 — the sign/scale convention of
flexOPT's residual) and the residual of the exact solution.

In [ ]:
homShell = layeredSHModel(; R=6371.0, radii=[3480.0, 6371.0], ρ=[3.3], Vsv=[4.5], Vsh=[4.5])
@show homShell.nzone homShell.solid_or_fluid outerSolidZones(homShell)
quick = shConvergenceHomogeneous(shConfigs; l=10, nIntervals=[100])

### 2b-ii — convergence, uniform h (default grid)

In [ ]:
convU = shConvergenceHomogeneous(shConfigs; l=10, nIntervals=[50, 100, 200, 400, 800])

### 2b-iii — convergence, planet1D DSM grid (`re`)
Same shell; h now follows GT95 eq. 6.2 with $\omega_{max}=\Omega/R$.

In [ ]:
convD = shConvergenceHomogeneous(shConfigs; l=10, grid=:dsm, reList=[1e-1, 3e-2, 1e-2, 3e-3, 1e-3])

In [ ]:
function plotSHConvergence(rows; title="")
    fig = Figure(size=(1000, 420))
    ax1 = Axis(fig[1, 1]; xscale=log10, yscale=log10, xlabel="h (km)",
               ylabel="max|A W| / max|A_Ω² W|", title="residual of the exact solution " * title)
    ax2 = Axis(fig[1, 2]; xscale=log10, yscale=log10, xlabel="h (km)",
               ylabel="‖A - sK‖/‖sK‖", title="distance to conventional DSM " * title)
    names = unique(getfield.(rows, :config))
    for nm in names
        rr = filter(r -> r.config == nm, rows)
        h = getfield.(rr, :h_km)
        scatterlines!(ax1, h, getfield.(rr, :resFlexOPT); label=nm)
        scatterlines!(ax2, h, getfield.(rr, :relDiffDSM); label=nm)
    end
    rr = filter(r -> r.config == first(names), rows)
    h = getfield.(rr, :h_km)
    lines!(ax1, h, getfield.(rr, :resDSM); color=:black, linewidth=3, label="DSM conventional")
    h0, r0 = h[end], rr[end].resDSM
    lines!(ax1, h, r0 .* (h ./ h0) .^ 2; color=:gray, linestyle=:dash, label="h²")
    lines!(ax1, h, r0 .* (h ./ h0) .^ 4; color=:gray, linestyle=:dot, label="h⁴")
    axislegend(ax1; position=:rb)
    axislegend(ax2; position=:rb)
    display(fig)
    return fig
end
plotSHConvergence(convU; title="(uniform h)")
plotSHConvergence(convD; title="(planet1D re-grid)")

In [ ]:
# Error constant relative to the conventional FD scheme, at the same h
function shErrorRatios(rows; reference="convFD3")
    for gp in unique(getfield.(rows, :gridParam))
        rr = filter(r -> r.gridParam == gp, rows)
        ref = only(filter(r -> r.config == reference, rr)).resFlexOPT
        @printf("h = %8.3f km   ", rr[1].h_km)
        for r in rr
            r.config == reference && continue
            @printf("%s/%s = %.3e   ", r.config, reference, r.resFlexOPT / ref)
        end
        @printf("DSMconv/%s = %.3e\n", reference, rr[1].resDSM / ref)
    end
end
println("uniform h:");      shErrorRatios(convU)
println("planet1D re-grid:"); shErrorRatios(convD)

### 2b-ii' — diagnostic: is the small-h floor an inexactness of the Δ=1 recipes?
Below ~15 km the ratios rise again (OPT5 first), i.e. an error growing as $h\to0$ — not truncation.
In the scaled coordinate the stiffness coefficients stay $O(1)$ while the physical term shrinks like $h^2$,
so a fixed recipe inexactness $\varepsilon$ appears as $\sim\varepsilon/(kh)^2$ in `res`.
This cell tests the recipes alone (unit constant material, polynomial $W=s^k$):
$\partial_s^2$ must kill $1, s$ exactly and reproduce $W''$ for $s^2, s^3$; the two first-order $Lr$ terms must cancel for every $W$.

In [ ]:
exactness = shStencilExactness(shConfigs)

### 2b-iv — real models, zone by zone
**(a)** a two-zone TI model with a discontinuity (Vsh ≠ Vsv in the upper zone), to see h change across the interface;
**(b)** PREM (outer solid region). No exact solution: `s/h` and the matrix distance to conventional DSM per zone.
$\Omega = 2\pi R/T$ with $T=100$ s.

In [ ]:
twoZoneTI = layeredSHModel(; R=6371.0, radii=[3480.0, 5701.0, 6371.0],
                             ρ=[5.0, 3.6], Vsv=[6.5, 4.6], Vsh=[6.5, 4.9])
gTwo = shRadialGrid(twoZoneTI; mode=:uniform, h=40.0)
for s in gTwo.segments
    @printf("zone %d: %7.1f – %7.1f km, %4d intervals, h = %.3f km\n", s.zone, s.r[1], s.r[end], length(s.r) - 1, s.h_km)
end
ΩT100 = 2π * 6371.0 / 100.0
cmpTwo = shZoneComparison(twoZoneTI, shConfigs; l=10, Ω=ΩT100, h=40.0)

In [ ]:
prem = planet1D.getSet1Dmodel!(:PREM)
gPrem = shRadialGrid(prem; mode=:uniform, h=25.0)
for s in gPrem.segments
    @printf("zone %2d: %7.1f – %7.1f km, %4d intervals, h = %7.3f km\n", s.zone, s.r[1], s.r[end], length(s.r) - 1, s.h_km)
end
cmpPrem = shZoneComparison(prem, shConfigs; l=10, Ω=ΩT100, h=25.0)

## ⏸ Stop and review (end of Step 2)

Please check / send back:
1. 2a: recipe construction OK for the four configs (form, `varM` has 5 rows: ρr², Lr², Lr, Nₗ, ω);
2. 2b-i: `s/h` ≈ constant (±1?) for the OPTs — if not, the scale/sign convention of the residual needs a look before going on;
3. 2b-ii/iii: `res` slopes (FD3, DSM, OPT3 all ~h²) and above all the ratios res(OPT)/res(convFD3) and res(DSM)/res(convFD3) at equal h; how OPT4/OPT5 behave;
4. 2b-iv: PREM zone table (thin crustal zones get a smaller h — `minIntervals=12`).

Next (Step 3): homogeneous sphere with free surface and a cut-off radius, forced problem and toroidal eigenfrequencies;
the boundary rows (and the modified operators, once the paper's equations are in hand).

---
# Step 3 — boundaries: traction-free shell, toroidal eigenfrequencies

Homogeneous isotropic mantle shell $r_{cmb}\le r\le R$, **traction-free at both ends** (SH sees the CMB as a free surface).
Exact: $W=A\,j_l(kr)+B\,y_l(kr)$ with $W'-W/r=0$ at both radii, i.e. $D(k)=g_j(ka)g_y(kb)-g_j(kb)g_y(ka)=0$,
$g_f(x)=x f_{l-1}(x)-(l+2)f_l(x)$, $\Omega=\beta k$. Discrete: $A(\Omega)=A_0+\Omega^2M$ ⇒ generalized eigenproblem $A_0v=-\Omega^2Mv$.

Rows near the ends (`helpers/sphericalTI_SH_step3.jl`):
* rows the scheme's stencil cannot reach (OPT4: 1 + 2, OPT5: 2 + 2, end rows included) → the non-end ones are **taken from OPT3** (first, simplest "overlap" of two recipes);
* the **end rows** get a closure:
  * `:dsmHalf` — Galerkin natural condition (conventional DSM half-element row, zero traction flux), rescaled to the scheme;
  * `(:traction, q)` — traction collocation $Lr^2\partial_sW-Lr\,W=0$ with one-sided Fornberg weights on $q{+}1$ nodes (no $\Omega^2$ part).

Python prototype with the DSM interior: DSM natural BC is a clean $h^2$; traction closure with $q=1$ is only $O(h)$, $q=2$ gives $h^2$, $q\ge3$ leaves the interior error dominant.
The question here: which closure keeps OPT3/OPT4/OPT5's interior accuracy?

In [ ]:
include(joinpath(flexopt_root, "helpers", "sphericalTI_SH_step3.jl"))
eigL10 = shShellEigenBenchmark(shConfigs; l=10, nIntervals=[50, 100, 200, 400],
                               closures=[:dsmHalf, (:traction, 2), (:traction, 3), (:traction, 4)])

In [ ]:
# lower angular order (stronger curvature terms relative to the stiffness)
eigL2 = shShellEigenBenchmark(shConfigs; l=2, nIntervals=[50, 100, 200, 400],
                              closures=[:dsmHalf, (:traction, 3), (:traction, 4)])

In [ ]:
function plotShellEigen(rows; modes=(1, 5), title="")
    fig = Figure(size=(1100, 450))
    for (ip, im) in enumerate(modes)
        ax = Axis(fig[1, ip]; xscale=log10, yscale=log10, xlabel="h (km)",
                  ylabel="|δΩ/Ω|", title="overtone n=$(im-1) " * title)
        for key in unique([(r.scheme, r.closure) for r in rows])
            rr = filter(r -> (r.scheme, r.closure) == key, rows)
            length(rr[1].relErr) >= im || continue
            h = getfield.(rr, :h_km)
            e = [abs(r.relErr[im]) for r in rr]
            ls = key[1] == "DSMconv" ? :solid : (occursin("traction", key[2]) ? :dash : :solid)
            scatterlines!(ax, h, max.(e, 1e-16); label="$(key[1]) $(key[2])", linestyle=ls)
        end
        ip == length(modes) && Legend(fig[1, ip+1], ax; framevisible=false, labelsize=10)
    end
    display(fig)
    return fig
end
plotShellEigen(eigL10; title="(l=10)")
plotShellEigen(eigL2; title="(l=2)")

## ⏸ Stop and review (end of Step 3a)
Things to look at (I will read them from the saved outputs):
1. no `⚠` lines (no spurious / missing modes), `nC` (complex eigenvalues from non-symmetric stencils) = 0;
2. which closure preserves each scheme's interior gain (OPT3 ≈ ×1/30 of FD3, OPT5 ≈ h⁴) — I expect `:dsmHalf` to cap OPT4/OPT5 at the DSM h² boundary error, and the traction closure with q ≥ 3–4 to do better;
3. l = 2 vs l = 10.

Next (Step 3b): the forced problem (point/body source at $r_s$, $f_W$ projection with the normalised $\mathbf C^*_{lm}$) and the cut-off radius for the whole-Earth case; then Step 4 = discontinuities with `overlapBoundaryLinearSystem`.